# Feature Extraction

This notebook loads the cleaned UrbanSound8K metadate from the previous notebook, extracts audio features, and saves them in ready-to-use files for the next notebook.

## Imports and Paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.feature_extraction import load_audio_fixed_length, extract_features

ROOT = Path.cwd().parent.resolve()

PROCESSED_DIR = ROOT / "data/processed"
FEATURE_DIR = ROOT / "data/features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

METADATA_DIR = PROCESSED_DIR / "urbansound8k_metadata_clean.csv"
CV_PLAN_DIR = PROCESSED_DIR / "cv_plan_10fold.json"

print("Metadata path:", METADATA_DIR)
print("Feature output dir:", FEATURE_DIR)

## Loading Cleaned Metadata

In [ ]:
metadata = pd.read_csv(METADATA_DIR)

print(metadata.shape)
display(metadata.head())

print("Classes:", sorted(metadata["class"].unique()))
print("Folds:", sorted(int(fold) for fold in metadata["fold"].unique()))

## Feature Extraction Parameters

In [ ]:
CONFIG = {
    "sample_rate": 22050,
    "duration": 4.0,
    "n_fft": 2048,
    "hop_length": 512,
    "n_mels": 128,
    "n_mfcc": 40,
}

CONFIG["target_length"] = int(CONFIG["sample_rate"] * CONFIG["duration"])
CONFIG

## Visual Sanity Check on One Audio Clip

In [ ]:
example_row = metadata.iloc[0]
example_path = example_row["audio_path"]

y = load_audio_fixed_length(
    example_path,
    sample_rate=CONFIG["sample_rate"],
    target_length=CONFIG["target_length"],
)

example_features, features_names, example_logmel = extract_features(
    example_path, CONFIG
)

example_mfcc = librosa.feature.mfcc(
    y=y,
    sr=CONFIG["sample_rate"],
    n_mfcc=CONFIG["n_mfcc"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
)

print("Example file:", example_row["slice_file_name"])
print("Class:", example_row["class"])
print("Waveform shape", y.shape)
print("Tabular feature vector shape:", example_features.shape)
print("Log-Mel shape:", example_logmel.shape)

In [ ]:
from librosa import example

plt.figure(figsize=(10, 3))
librosa.display.waveshow(y, sr=CONFIG["sample_rate"])
plt.title(f"Waveform example: {example_row['class']}")
plt.xlabel("Time, seconds")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
librosa.display.specshow(
    example_logmel,
    sr=CONFIG["sample_rate"],
    hop_length=CONFIG["hop_length"],
    x_axis="time",
    y_axis="mel",
)
plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram Example")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
librosa.display.specshow(
    example_mfcc,
    sr=CONFIG["sample_rate"],
    hop_length=CONFIG["hop_length"],
    x_axis="time",
)
plt.colorbar()
plt.title("MFCC Example")
plt.ylabel("MFCC coefficient")
plt.tight_layout()
plt.show()

## Extracting Features for All Clips

In [ ]:
X_tabular = []
X_logmel = []
labels = []
class_ids = []
folds = []
filenames = []

feature_names = None

for _, row in tqdm(
    metadata.iterrows(), total=len(metadata), desc="Extracting features"
):
    features, names, logmel = extract_features(row["audio_path"], CONFIG)

    if feature_names is None:
        feature_names = names

    X_tabular.append(features)
    X_logmel.append(logmel)
    labels.append(row["class"])
    class_ids.append(row["classID"])
    folds.append(row["fold"])
    filenames.append(row["slice_file_name"])

X_tabular = np.asarray(X_tabular, dtype=np.float32)
X_logmel = np.asarray(X_logmel, dtype=np.float32)
labels = np.asarray(labels)
class_ids = np.asarray(class_ids, dtype=np.int64)
folds = np.asarray(folds, dtype=np.int64)
filenames = np.asarray(filenames)

print("Tabular features:", X_tabular.shape)
print("Log-Mel spectrograms:", X_logmel.shape)
print("Labels:", labels.shape)
print("Feature names:", len(feature_names))

## Checking for Invalid Values

In [ ]:
print("NaN in tabular features:", np.isnan(X_tabular).sum())
print("Inf in tabular features:", np.isinf(X_tabular).sum())

print("NaN in log-Mel:", np.isnan(X_logmel).sum())
print("Inf in log-Mel:", np.isinf(X_logmel).sum())

## Saving Features for the Next Notebook

In [ ]:
tabular_path = FEATURE_DIR / "tabular_audio_features.npz"

np.savez_compressed(
    tabular_path,
    X=X_tabular,
    y=class_ids,
    labels=labels,
    folds=folds,
    filenames=filenames,
    feature_names=np.asarray(feature_names),
)

print("Saved:", tabular_path)

In [ ]:
logmel_path = FEATURE_DIR / "logmel_spectrogram.npz"

np.savez_compressed(
    logmel_path,
    X=X_logmel,
    y=class_ids,
    labels=labels,
    folds=folds,
    filenames=filenames,
)

print("Saved:", logmel_path)

In [ ]:
feature_config_path = FEATURE_DIR / "feature_config.json"

with open(feature_config_path, "w") as f:
    json.dump(CONFIG, f, indent=4)

print("Saved:", feature_config_path)

## Final Verification

In [ ]:
tabular_data = np.load(tabular_path, allow_pickle=True)
logmel_data = np.load(logmel_path, allow_pickle=True)

print("Tabular keys:", tabular_data.files)
print("Log-Mel keys:", logmel_data.files)

print("Reloaded tabular X:", tabular_data["X"].shape)
print("Reloaded Log-Mel X:", logmel_data["X"].shape)
print("Reloaded y:", tabular_data["y"].shape)
print("Reloaded folds:", tabular_data["folds"].shape)